---
# AlphaGenome PyTorch (Community Port)

Source: https://github.com/genomicsxai/alphagenome-pytorch

This section uses the **PyTorch** port of AlphaGenome. Because the original model
processes 1 Mb by splitting it into **8 chunks of 131,072 bp** (`2¹⁷`) in parallel
across TPUs, this PyTorch implementation handles **one chunk at a time** to fit
within single-GPU memory (~14 GB).

The `encode()` call returns a dict with two keys:
- **`embeddings_128bp`** — shape `(batch, seq_positions, 3072)`: contextual
  embeddings at 128 bp resolution (1,024 positions per chunk).
- **`embeddings_pair`** — pairwise contact-map embeddings.

Mean-pooling `embeddings_128bp` over the sequence dimension (`dim=1`) produces a
single `(batch, 3072)` vector summarising the genomic region — suitable as input
to downstream classifiers or similarity search.

---

In [ ]:
!pip install pyfaidx umap-learn tqdm

In [ ]:
# 1. Download hg38 from Ensembl
!wget -N -O hg38.fa.gz https://ftp.ensembl.org/pub/release-110/fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz

# 2. Decompress (using -f to overwrite if an old partial file exists)
!gunzip -f hg38.fa.gz

# 3. Find the decompressed file and rename it immediately to a standard name
!mv Homo_sapiens.GRCh38.dna.primary_assembly.fa hg38.fa 2>/dev/null || mv *.fa hg38.fa

# 4. Standardize naming: Ensembl (1, 2, X) -> UCSC (chr1, chr2, chrX) for compatibility
!sed -i 's/^>/>chr/' hg38.fa
!sed -i 's/^>chrMT/>chrM/' hg38.fa

print("Genome downloaded and renamed. Chromosome names standardized to 'chr' format.")

In [ ]:
!pip install alphagenome-pytorch

In [ ]:
# ── Download pretrained weights from HuggingFace ──────────────────────────────
!python -c "from huggingface_hub import hf_hub_download; hf_hub_download('gtca/alphagenome_pytorch', 'model_all_folds.safetensors', local_dir='.')"

In [ ]:
# ── Convert safetensors → .pt ─────────────────────────────────────────────────
import os
from safetensors.torch import load_file
import torch

# Guard: ensure the download completed before proceeding
assert os.path.exists("model_all_folds.safetensors"), (
    "Weights file not found — re-run the download cell above."
)

weights = load_file("model_all_folds.safetensors")
torch.save(weights, "alphagenome.pt")
print("Saved alphagenome.pt")

In [ ]:
# ── Imports (PyTorch section) ─────────────────────────────────────────────────
import os
import torch
import numpy as np
from alphagenome_pytorch import AlphaGenome
from alphagenome_pytorch.utils.sequence import sequence_to_onehot_tensor

# ── Device: fall back to CPU if CUDA is unavailable ──────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

In [ ]:
# ── Load pretrained model ─────────────────────────────────────────────────────
assert os.path.exists("alphagenome.pt"), (
    "alphagenome.pt not found — run the conversion cell above."
)

model = AlphaGenome.from_pretrained('alphagenome.pt', device=device)
model.eval()

In [ ]:
# ── Sequence size constants & Config ──────────────────────────────────────────
CONTEXT_SIZE = 2**20  # 1,048,576 bp
CHUNK_SIZE   = CONTEXT_SIZE // 8  # 131,072 bp

# Config for whole-genome processing
BATCH_SIZE = 4  # Lower to 2 if you get "CUDA out of memory"
MAX_N_TOLERANCE = 0.10  # Skip chunks with more than 10% N bases (assembly gaps)

print(f"CHUNK_SIZE: {CHUNK_SIZE:,} bp")


In [ ]:
# ── Whole-Genome Inference Loop (RESUMABLE) ──────────────────────────────────
import os
from pyfaidx import Fasta
from tqdm.auto import tqdm
import gc
import pandas as pd
import numpy as np
import torch

# 1. Output directory setup
OUTPUT_DIR = "hg38_atlas"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. Load genome
genome = Fasta('hg38.fa')
chromosomes = [f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY"]

def process_batch(seqs):
    """Convert sequences to tensors, run through AlphaGenome, return pooled vectors."""
    tensors = [sequence_to_onehot_tensor(s.upper()) for s in seqs]
    batch_tensor = torch.stack(tensors).to(device)
    org_idx = torch.zeros(len(seqs), dtype=torch.long, device=device)

    with torch.inference_mode():
        emb = model.encode(batch_tensor, organism_index=org_idx, resolutions=(128,))
        pooled = emb['embeddings_128bp'].mean(dim=1).cpu().numpy()

    del batch_tensor, emb
    torch.cuda.empty_cache()
    return pooled

print(f"Starting scan. Results will be saved to: '{OUTPUT_DIR}'")

# 3. MAIN LOOP (per chromosome)
for chrom in chromosomes:
    # --- RESUME LOGIC ---
    emb_path  = f"{OUTPUT_DIR}/{chrom}_embeddings.npy"
    meta_path = f"{OUTPUT_DIR}/{chrom}_metadata.csv"

    if os.path.exists(emb_path) and os.path.exists(meta_path):
        print(f"✅ {chrom} already processed, skipping.")
        continue
    # --------------------

    print(f"⏳ Processing {chrom}...")
    chr_seq = genome[chrom]
    chr_len = len(chr_seq)

    chr_embeddings = []
    chr_metadata   = []
    batch_seqs     = []
    batch_meta     = []

    # Split chromosome into windows
    for start in tqdm(range(0, chr_len - CHUNK_SIZE, CHUNK_SIZE), desc=f"Chunks in {chrom}"):
        end     = start + CHUNK_SIZE
        seq_str = str(chr_seq[start:end])

        # Skip chunks with too many N bases
        if (seq_str.upper().count('N') / CHUNK_SIZE) > MAX_N_TOLERANCE:
            continue

        batch_seqs.append(seq_str)
        batch_meta.append({'chrom': chrom, 'start': start, 'end': end})

        # Process batch
        if len(batch_seqs) == BATCH_SIZE:
            embs = process_batch(batch_seqs)
            chr_embeddings.append(embs)
            chr_metadata.extend(batch_meta)
            batch_seqs, batch_meta = [], []

    # Process remaining chunks
    if batch_seqs:
        embs = process_batch(batch_seqs)
        chr_embeddings.append(embs)
        chr_metadata.extend(batch_meta)

    # --- SAVE CHROMOSOME RESULTS ---
    if chr_embeddings:
        chr_matrix = np.vstack(chr_embeddings)
        chr_df     = pd.DataFrame(chr_metadata)

        np.save(emb_path, chr_matrix)
        chr_df.to_csv(meta_path, index=False)
        print(f"💾 Saved: {chrom} ({chr_matrix.shape[0]} valid windows)")
    else:
        print(f"⚠️ {chrom} produced no valid data (too many N bases).")

    # Free memory before next chromosome
    gc.collect()

print("🎉 All chromosomes scanned successfully!")


In [ ]:
# ── Load the full atlas for visualization or clustering ───────────────────────
import glob

# Find all output files
npy_files = sorted(glob.glob(f"{OUTPUT_DIR}/*_embeddings.npy"))
csv_files = sorted(glob.glob(f"{OUTPUT_DIR}/*_metadata.csv"))

print(f"Found {len(npy_files)} processed chromosomes.")

# Load and concatenate
matrices   = [np.load(f) for f in npy_files]
dfs        = [pd.read_csv(f) for f in csv_files]

embeddings = np.vstack(matrices)
metadata   = pd.concat(dfs, ignore_index=True)

print(f"Atlas loaded. Total shape: {embeddings.shape}")
